In [ ]:
#| default_exp topology

In [ ]:
#| export

from __future__ import annotations

from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Dict, Iterable, List, Tuple

from comm_core.core import NodeId


@dataclass
class Link:
    sender: NodeId
    receiver: NodeId

    bandwidth_hz: float = float("inf")
    frequency_hz: float = 0.0
    tx_power_w: float = 1.0

    enabled: bool = True



In [ ]:
#| export
class Topology(ABC):

    @abstractmethod
    def neighbors(self, node: NodeId) -> List[NodeId]:
        pass

    @abstractmethod
    def is_connected(
        self,
        sender: NodeId,
        receiver: NodeId,
    ) -> bool:
        pass

    @abstractmethod
    def get_link(
        self,
        sender: NodeId,
        receiver: NodeId,
    ) -> Link:
        pass



In [ ]:
#| export
class GraphTopology(Topology):

    def __init__(
        self,
        links: Iterable[Link],
    ):
        self._links: Dict[Tuple[NodeId, NodeId], Link] = {
            (link.sender, link.receiver): link
            for link in links
        }

    def neighbors(self, node: NodeId) -> List[NodeId]:
        return [
            receiver
            for (sender, receiver), link in self._links.items()
            if sender == node and link.enabled
        ]

    def is_connected(
        self,
        sender: NodeId,
        receiver: NodeId,
    ) -> bool:
        link = self._links.get((sender, receiver))
        return link is not None and link.enabled

    def get_link(
        self,
        sender: NodeId,
        receiver: NodeId,
    ) -> Link:
        try:
            return self._links[(sender, receiver)]
        except KeyError:
            raise ValueError(
                f"No link from {sender} to {receiver}"
            )



In [ ]:
#| export

class FullMeshTopology(GraphTopology):

    def __init__(
        self,
        nodes: Iterable[NodeId],
    ):
        nodes = list(nodes)

        links = [
            Link(sender=a, receiver=b)
            for a in nodes
            for b in nodes
            if a != b
        ]

        super().__init__(links)



In [ ]:
#| export

class StarTopology(GraphTopology):

    def __init__(
        self,
        center: NodeId,
        nodes: Iterable[NodeId],
    ):
        links = []

        for node in nodes:
            if node == center:
                continue

            links.append(
                Link(center, node)
            )
            links.append(
                Link(node, center)
            )

        super().__init__(links)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()